# 13.6 · 流模型 / Normalizing Flows (RealNVP)

> **课程定位 / Where this fits**
> 第 6 课，**Part 13 · 生成模型**。第三类生成范式, 独门绝技是"**精确似然**"。
> Lesson 6, **Part 13 · Generative Models**. A third paradigm whose superpower is "**exact likelihood**."
>
> VAE 用近似、GAN 用对抗, 都**算不出数据的精确概率**。**流模型(normalizing flow)** 走一条数学上很优雅的路：用**一系列可逆变换**, 把一个简单分布(标准高斯)精确地"掰"成复杂的数据分布。因为每步都可逆、且能算雅可比行列式, 它能**精确计算任意数据点的似然 $p(x)$**(VAE 只能给下界, GAN 完全给不了)。本课讲清**变量变换公式**与 **RealNVP 的耦合层**, 在 2D 玩具数据上从零实现一个流, **可视化"高斯↔数据"的双向变换**。
> VAE approximates, GAN uses adversarial training — neither **computes the exact probability** of data. **Normalizing flows** take a mathematically elegant route: a **series of invertible transformations** reshape a simple distribution (standard Gaussian) exactly into the complex data distribution. Because each step is invertible with a computable Jacobian, it can **exactly compute the likelihood $p(x)$** of any point (VAE only bounds it, GAN can't at all). We cover the **change-of-variables formula** and **RealNVP coupling layers**, implement a flow from scratch on 2D toy data, and **visualize the bidirectional Gaussian↔data transform**.
>
> 💼 **实战/面试视角**："流模型为什么能精确似然 / 变量变换公式 / 耦合层为什么可逆且雅可比好算 / 三类生成模型对比" 是生成模型进阶题。
> 💼 **Practical/interview angle:** "why flows give exact likelihood / change of variables / why coupling layers are invertible with easy Jacobian / comparing the three families" — advanced.

> 📐 **符号约定 / Notation**
> - 基分布 $p_z$ —— 简单分布(标准高斯) / base distribution (standard Gaussian)
> - $f$ —— 可逆变换, $z = f(x)$ / invertible transform
> - 雅可比行列式 $\det J$ —— 变换的"体积缩放" / Jacobian determinant (volume scaling)

> 💡 **面试相关 / Interview-relevant**
> - "流模型如何精确计算似然(变量变换)"（出镜率 ★★★★）
> - "耦合层(coupling)为什么可逆、雅可比为什么三角"（★★★★）
> - "VAE vs GAN vs Flow(似然/质量/采样速度)"（★★★★★）
> - "流模型的限制(可逆约束/维度不变)"（★★★）

---

## 学习目标 / Learning Objectives
1. 理解流模型用可逆变换把高斯掰成数据分布, 能精确算似然。
   Understand flows reshape a Gaussian into data via invertible maps, with exact likelihood.
2. 掌握**变量变换公式**与为什么需要雅可比行列式。
   Master the change-of-variables formula and why the Jacobian appears.
3. 理解 **RealNVP 耦合层**为何可逆且雅可比易算。
   Understand why RealNVP coupling layers are invertible with an easy Jacobian.
4. **从零实现流**, 可视化高斯↔数据双向变换并采样生成。
   Implement a flow from scratch, visualize the Gaussian↔data transform, and sample.

## 目录 / TOC
1. [流模型与精确似然 ⭐](#1)
2. [变量变换与耦合层 ⭐](#2)
3. [从零实现 RealNVP ⭐](#3)
4. [三类生成模型对比 + 小结 ⭐](#4)


<a id="1"></a>
## 1. 流模型与精确似然 ⭐ / Flows & Exact Likelihood

核心想法：我们想要的复杂数据分布 $p(x)$ 很难直接写出来；但一个**标准高斯 $p(z)$** 简单到能精确计算。流模型学一个**可逆变换 $f$**, 把数据 $x$ 变成高斯里的点 $z = f(x)$(反过来 $x = f^{-1}(z)$ 就能从高斯采样生成数据)。
The core idea: the complex data distribution $p(x)$ is hard to write directly; but a **standard Gaussian $p(z)$** is simple enough to compute exactly. A flow learns an **invertible map $f$** that turns data $x$ into Gaussian points $z = f(x)$ (and inversely $x = f^{-1}(z)$ generates data by sampling the Gaussian).

**为什么能精确算似然**(流模型的独门优势)：只要 $f$ **可逆**, 概率论的**变量变换公式**就能把"复杂的 $p(x)$"用"简单的 $p(z)$"精确表达出来(下一节)。于是流模型能直接最大化训练数据的对数似然 $\log p(x)$ ——**有精确、明确的训练目标**, 不像 GAN 那样靠对抗、也不像 VAE 那样只能优化下界。
**Why exact likelihood** (the flow's unique edge): as long as $f$ is **invertible**, the **change-of-variables formula** expresses the complex $p(x)$ exactly via the simple $p(z)$ (next section). So a flow directly maximizes the data log-likelihood $\log p(x)$ — a **precise, explicit objective**, unlike GAN's adversarial game or VAE's lower bound.

代价(面试点)：变换必须**处处可逆**, 这是个强约束——限制了网络结构(不能随便用任意网络), 且**输入输出维度必须相同**(不能像 VAE 那样降维)。所以流模型参数效率较低、对高分辨率图像较吃力。
The cost (interview): the map must be **invertible everywhere**, a strong constraint — limiting architectures (can't use arbitrary nets) and forcing **equal input/output dimensions** (no dimensionality reduction like VAE). So flows are less parameter-efficient and struggle with high-res images.


<a id="2"></a>
## 2. 变量变换与耦合层 ⭐ / Change of Variables & Coupling Layers

**变量变换公式(change of variables)**——流模型的数学核心：
The **change-of-variables formula** — the math core of flows:

$$\log p(x) = \log p_z(f(x)) + \log\left|\det \frac{\partial f}{\partial x}\right|$$

直觉：第一项是"变换后的点在高斯下有多可能"；第二项**雅可比行列式**修正"变换把空间的体积拉伸/压缩了多少"(把概率密度搬来搬去时, 体积变了密度也要相应变)。
Intuition: the first term is "how likely the transformed point is under the Gaussian"; the second, the **Jacobian determinant**, corrects for "how much the transform stretches/squeezes volume" (moving densities around changes volume, hence density).

**难点**：一般的雅可比行列式计算量是 $O(d^3)$, 太贵。**RealNVP 的耦合层(affine coupling)** 巧妙地让雅可比变成**三角矩阵**(行列式 = 对角线之积, 极易算), 同时保证可逆。做法：
**The catch:** a general Jacobian determinant costs $O(d^3)$, too expensive. **RealNVP's affine coupling layer** cleverly makes the Jacobian **triangular** (determinant = product of the diagonal, trivial) while staying invertible. How:
- 把输入维度**分成两半** $x_1, x_2$。
  Split the input dimensions into halves $x_1, x_2$.
- **$x_1$ 原样保留**；用 $x_1$ 算出缩放 $s$ 和平移 $t$, 去**仿射变换 $x_2$**: $y_2 = x_2 \cdot e^{s(x_1)} + t(x_1)$。
  **Keep $x_1$ unchanged;** from $x_1$ compute scale $s$ and shift $t$, then **affine-transform $x_2$**: $y_2 = x_2 \cdot e^{s(x_1)} + t(x_1)$.
- **可逆**：已知 $y_1=x_1$, 就能反解 $x_2 = (y_2 - t)\cdot e^{-s}$。**雅可比是三角的**, log-det = $\sum s$(超好算)。
  **Invertible:** knowing $y_1=x_1$ recovers $x_2 = (y_2 - t)\cdot e^{-s}$. The **Jacobian is triangular**, log-det = $\sum s$ (trivial).
- **交替**哪一半被变换, 堆叠多层 → 所有维度都被充分混合变换。
  **Alternate** which half is transformed across stacked layers → all dimensions get thoroughly transformed.

> 妙处:$s, t$ 可以是**任意复杂的神经网络**(因为求逆只需要它们的输出, 不需要对 $s,t$ 本身求逆)! 这让耦合层既灵活又可逆。
> The beauty: $s, t$ can be **arbitrarily complex neural nets** (inverting only needs their outputs, not inverting $s,t$ themselves)! So coupling layers are both flexible and invertible.


<a id="3"></a>
## 3. 从零实现 RealNVP ⭐ / RealNVP From Scratch

在 **2D 双月牙(two-moons)** 玩具数据上实现 RealNVP(2D 便于可视化"高斯↔数据"的变换)。训练目标：最大化数据的对数似然(= 最小化负对数似然 NLL, 用上面的公式)。
Implement RealNVP on **2D two-moons** toy data (2D for visualizing the Gaussian↔data transform). Objective: maximize data log-likelihood (= minimize NLL via the formula above).


In [ ]:
import numpy as np, matplotlib.pyplot as plt, seaborn as sns, time
import torch, torch.nn as nn
from sklearn.datasets import make_moons
sns.set_theme(style="white"); torch.manual_seed(0); np.random.seed(0)

Xraw, _ = make_moons(4000, noise=0.05)
X = torch.tensor((Xraw - Xraw.mean(0)) / Xraw.std(0), dtype=torch.float32)   # 标准化 / standardize

class AffineCoupling(nn.Module):
    def __init__(self, mask):
        super().__init__(); self.register_buffer("mask", mask)               # mask 决定哪一半保留 / which half is kept
        self.st = nn.Sequential(nn.Linear(2,64), nn.ReLU(), nn.Linear(64,64), nn.ReLU(), nn.Linear(64,4))  # 算 s,t 的网络 / net for s,t
    def forward(self, x):                                                    # 数据→潜空间 / data → latent
        xm = x * self.mask                                                   # 保留的一半 / kept half
        h = self.st(xm); s, t = h[:,:2], h[:,2:]
        s = torch.tanh(s) * (1-self.mask); t = t * (1-self.mask)             # 只变换另一半 / transform the other half
        y = xm + (1-self.mask) * (x * torch.exp(s) + t)                      # 仿射: x2·e^s + t / affine
        return y, s.sum(1)                                                   # 返回 + log-det(=Σs) / + log-det
    def inverse(self, y):                                                    # 潜空间→数据 / latent → data
        ym = y * self.mask; h = self.st(ym); s, t = h[:,:2], h[:,2:]
        s = torch.tanh(s) * (1-self.mask); t = t * (1-self.mask)
        return ym + (1-self.mask) * ((y - t) * torch.exp(-s))                # 反解 x2=(y2-t)e^{-s} / invert

class RealNVP(nn.Module):
    def __init__(self, n_layers=6):
        super().__init__()
        masks = [torch.tensor([1.,0.]), torch.tensor([0.,1.])]               # 交替遮哪一半 / alternate masks
        self.layers = nn.ModuleList([AffineCoupling(masks[i%2]) for i in range(n_layers)])
    def forward(self, x):                                                    # data → z, 累加 log-det / accumulate log-det
        ld = 0
        for l in self.layers: x, d = l(x); ld = ld + d
        return x, ld
    def inverse(self, z):                                                    # z → data (反序逆变换) / reverse
        for l in reversed(self.layers): z = l.inverse(z)
        return z

flow = RealNVP(); opt = torch.optim.Adam(flow.parameters(), 1e-3); t0 = time.time()
for step in range(2000):
    xb = X[torch.randint(0, len(X), (512,))]
    z, logdet = flow(xb)                                                     # data → latent
    # NLL = -log p(x) = -[log N(z) + log-det]; log N(z) = -0.5 z² - log(2π) (逐维求和) / change-of-variables
    nll = (0.5*(z**2).sum(1) - logdet + np.log(2*np.pi)).mean()
    opt.zero_grad(); nll.backward(); opt.step()
print(f"RealNVP 训练完成 ({time.time()-t0:.0f}s), 负对数似然 NLL = {nll.item():.3f} (这是精确似然!)")

# 三联图: 真实数据 / 流生成的样本 / 数据被映射到的潜空间(应≈高斯) / three panels
with torch.no_grad():
    gen = flow.inverse(torch.randn(2000, 2)).numpy()                         # 从高斯采样→生成数据 / sample → data
    z_of_data, _ = flow(X[:2000]); z_of_data = z_of_data.numpy()             # 数据→潜空间 / data → latent
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].scatter(X[:2000,0], X[:2000,1], s=4, alpha=0.5, c="#39c"); axes[0].set_title("真实数据 (two-moons)")
axes[1].scatter(gen[:,0], gen[:,1], s=4, alpha=0.5, c="#e76f51"); axes[1].set_title("流生成: 高斯采样→逆变换→新数据")
axes[2].scatter(z_of_data[:,0], z_of_data[:,1], s=4, alpha=0.5, c="#2a9d8f"); axes[2].set_title("数据→潜空间 (应≈标准高斯)")
for a in axes: a.set_xlim(-3,3); a.set_ylim(-3,3); a.set_aspect("equal")
plt.tight_layout(); plt.show()
print("中图: 从高斯随便采点, 逆变换→生成出月牙形数据(像真的); 右图: 月牙数据被正变换'掰直'成高斯团")
print("流双向可逆: data↔gaussian; 训练就是把数据似然(精确可算)最大化")


<a id="4"></a>
## 4. 三类生成模型对比 + 小结 ⭐ / Comparing the Three Families

到此我们见过三大类生成模型, 面试最爱让你对比(各有取舍, 没有绝对赢家)：
We've now seen three families; interviewers love the comparison (each has trade-offs, no absolute winner):

| | VAE | GAN | 流模型 Flow |
|---|---|---|---|
| 原理 | 编码成分布+重建(ELBO) | 生成器vs判别器对抗 | 可逆变换+变量变换 |
| 似然 | 只能下界(近似) | 完全没有 | **精确** |
| 生成质量 | 偏模糊 | **锐利** | 中等 |
| 采样速度 | 快(一次解码) | **快**(一次前向) | 快(一次逆变换) |
| 训练 | 稳定 | 不稳定(模式坍塌) | 稳定 |
| 隐空间 | 有, 平滑 | 有(但无显式似然) | 有, 可逆 |
| 限制 | 模糊 | 难训/无似然 | 维度不变+可逆约束 |

**流模型的定位**：在需要**精确似然**(异常检测、密度估计、可逆性)的场景很有价值；著名的 **Glow** 把流用到了人脸生成。但在纯图像质量上, 它不如 GAN/扩散。
**Flows' niche:** valuable when you need **exact likelihood** (anomaly detection, density estimation, invertibility); the famous **Glow** scaled flows to face generation. But on pure image quality, they trail GANs/diffusion.

```
流模型: 用一串可逆变换把简单高斯掰成复杂数据分布; 独门优势=能精确算似然p(x)
变量变换: log p(x)=log p_z(f(x))+log|det J|; 雅可比行列式修正体积缩放
RealNVP耦合层: 分两半, 保留一半+用它仿射另一半(x2·e^s+t); 可逆+雅可比三角(log-det=Σs); 交替堆叠
妙处: s,t可以是任意复杂网络(求逆只需其输出); 限制: 维度不变+处处可逆
三类对比: VAE(模糊/稳/似然下界) GAN(锐利/难训/无似然) Flow(精确似然/中等质量/维度不变)
应用: Glow(人脸)/密度估计/异常检测; 纯图像质量不如GAN/扩散
```

### 💡 面试速查 / Interview cheat-sheet
1. **流模型**: 可逆变换把高斯↔数据; 独门=精确似然(VAE只下界, GAN无)。
   Flows: invertible map Gaussian↔data; unique = exact likelihood.
2. **变量变换**: log p(x)=log p_z(z)+log|det J|; 雅可比修正体积。
   Change of variables: log p(x)=log p_z(z)+log|det J|; Jacobian corrects volume.
3. **耦合层**: 分两半, 一半仿射变另一半→可逆+雅可比三角(log-det=Σs)。
   Coupling: split halves, affine one by the other → invertible + triangular Jacobian.
4. **限制**: 维度必须不变 + 处处可逆 → 参数效率低/高分辨率吃力。
   Limits: dimension-preserving + invertible → less efficient/struggles at high-res.
5. **三类对比**: VAE模糊稳, GAN锐利难训, Flow精确似然。
   Three families: VAE blurry/stable, GAN sharp/unstable, Flow exact-likelihood.

### 下一节 / Next
**13.7 扩散模型(Diffusion)**——当今图像生成的**王者**(DALL·E 2、Stable Diffusion、Midjourney 都基于它)。思路出奇简单又强大: **逐步给图加噪声直到变成纯噪声, 再训练一个网络逐步去噪**——生成时从纯噪声出发, 一步步"去噪"出清晰图像。我们会从零实现 DDPM 并在 MNIST 上生成数字。
**13.7 Diffusion** — today's **king** of image generation (DALL·E 2, Stable Diffusion, Midjourney). Surprisingly simple yet powerful: **gradually add noise to an image until it's pure noise, then train a network to gradually denoise** — at generation, start from pure noise and "denoise" step by step into a clear image. We'll implement DDPM from scratch and generate MNIST digits.
